In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = r"C:\Users\HP\Internship\DataCoSupplyChainDataset.csv"

df = pd.read_csv(file_path, encoding="cp1252")

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Statu

In [2]:
required_columns = [
    "order date (DateOrders)",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Category Name"
]

for col in required_columns:
    print(col, ":", col in df.columns)

order date (DateOrders) : True
Days for shipping (real) : True
Days for shipment (scheduled) : True
Category Name : True


In [3]:
df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

df["Days for shipping (real)"] = pd.to_numeric(
    df["Days for shipping (real)"],
    errors="coerce"
)

df["Days for shipment (scheduled)"] = pd.to_numeric(
    df["Days for shipment (scheduled)"],
    errors="coerce"
)

df["Weekday"] = df["order date (DateOrders)"].dt.day_name()

df["Delay Days"] = (
    df["Days for shipping (real)"]
    - df["Days for shipment (scheduled)"]
)

df["Is Late"] = (
    df["Delay Days"] > 0
).astype(int)

df["Is On Time"] = (
    df["Delay Days"] <= 0
).astype(int)

print(df[[
    "Weekday",
    "Category Name",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delay Days",
    "Is Late"
]].head())

     Weekday   Category Name  Days for shipping (real)  \
0  Wednesday  Sporting Goods                         3   
1   Saturday  Sporting Goods                         5   
2   Saturday  Sporting Goods                         4   
3   Saturday  Sporting Goods                         3   
4   Saturday  Sporting Goods                         2   

   Days for shipment (scheduled)  Delay Days  Is Late  
0                              4          -1        0  
1                              4           1        1  
2                              4           0        0  
3                              4          -1        0  
4                              4          -2        0  


In [4]:
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_analysis = (
    df.groupby("Weekday")
    .agg(
        Total_Records=("Weekday", "size"),
        Avg_Delivery_Days=("Days for shipping (real)", "mean"),
        Avg_Scheduled_Days=("Days for shipment (scheduled)", "mean"),
        Avg_Delay_Days=("Delay Days", "mean"),
        Late_Deliveries=("Is Late", "sum"),
        On_Time_or_Early=("Is On Time", "sum")
    )
    .reindex(weekday_order)
    .reset_index()
)

weekday_analysis["Late_Delivery_Percentage"] = (
    weekday_analysis["Late_Deliveries"]
    / weekday_analysis["Total_Records"] * 100
).round(2)

weekday_analysis = weekday_analysis.round(2)

weekday_analysis

,Weekday,Total_Records,Avg_Delivery_Days,Avg_Scheduled_Days,Avg_Delay_Days,Late_Deliveries,On_Time_or_Early,Late_Delivery_Percentage
0,Monday,25786,3.52,2.94,0.58,14938,10848,57.93
1,Tuesday,25622,3.49,2.94,0.54,14503,11119,56.60
2,Wednesday,25587,3.51,2.94,0.56,14633,10954,57.19
3,Thursday,25752,3.49,2.92,0.57,14740,11012,57.24
4,Friday,25925,3.49,2.92,0.57,14914,11011,57.53
5,Saturday,25901,3.51,2.94,0.56,14748,11153,56.94
6,Sunday,25946,3.49,2.92,0.57,14924,11022,57.52


In [5]:
category_analysis = (
    df.groupby("Category Name")
    .agg(
        Total_Records=("Category Name", "size"),
        Avg_Delivery_Days=("Days for shipping (real)", "mean"),
        Avg_Scheduled_Days=("Days for shipment (scheduled)", "mean"),
        Avg_Delay_Days=("Delay Days", "mean"),
        Late_Deliveries=("Is Late", "sum"),
        On_Time_or_Early=("Is On Time", "sum")
    )
    .reset_index()
)

category_analysis["Late_Delivery_Percentage"] = (
    category_analysis["Late_Deliveries"]
    / category_analysis["Total_Records"] * 100
).round(2)

category_analysis = category_analysis.round(2)

category_analysis = category_analysis.sort_values(
    "Late_Delivery_Percentage",
    ascending=False
)

category_analysis

,Category Name,Total_Records,Avg_Delivery_Days,Avg_Scheduled_Days,Avg_Delay_Days,Late_Deliveries,On_Time_or_Early,Late_Delivery_Percentage
23,Golf Bags & Carts,61,3.34,2.57,0.77,42,19,68.85
32,Lacrosse,343,3.62,2.96,0.66,213,130,62.10
8,Cameras,592,3.34,2.69,0.65,367,225,61.99
37,Pet Supplies,492,3.39,2.68,0.71,302,190,61.38
40,Sporting Goods,357,3.49,2.87,0.62,214,143,59.94
4,Basketball,67,3.46,2.99,0.48,40,27,59.70
19,Fitness Accessories,309,3.57,2.96,0.61,184,125,59.55
15,Crafts,484,3.58,2.94,0.63,288,196,59.50
41,Strength Training,111,3.75,3.08,0.67,66,45,59.46
36,Music,434,3.48,2.91,0.57,258,176,59.45


In [6]:
weekday_category_analysis = (
    df.groupby(["Weekday", "Category Name"])
    .agg(
        Total_Records=("Category Name", "size"),
        Avg_Delivery_Days=("Days for shipping (real)", "mean"),
        Avg_Scheduled_Days=("Days for shipment (scheduled)", "mean"),
        Avg_Delay_Days=("Delay Days", "mean"),
        Late_Deliveries=("Is Late", "sum")
    )
    .reset_index()
)

weekday_category_analysis["Late_Delivery_Percentage"] = (
    weekday_category_analysis["Late_Deliveries"]
    / weekday_category_analysis["Total_Records"] * 100
).round(2)

weekday_category_analysis = weekday_category_analysis.round(2)

weekday_category_analysis.head(20)

,Weekday,Category Name,Total_Records,Avg_Delivery_Days,Avg_Scheduled_Days,Avg_Delay_Days,Late_Deliveries,Late_Delivery_Percentage
0,Friday,Accessories,247,3.52,2.96,0.56,138,55.87
1,Friday,As Seen on TV!,10,4.40,3.20,1.20,7,70.00
2,Friday,Baseball & Softball,104,3.37,2.76,0.61,58,55.77
3,Friday,Basketball,5,3.80,4.00,-0.20,2,40.00
4,Friday,Books,80,3.25,2.70,0.55,48,60.00
5,Friday,Boxing & MMA,53,3.75,2.57,1.19,38,71.70
6,Friday,Cameras,68,3.32,2.29,1.03,48,70.59
7,Friday,Camping & Hiking,1944,3.48,2.96,0.53,1101,56.64
8,Friday,Cardio Equipment,1773,3.52,2.93,0.60,1037,58.49
9,Friday,Children's Clothing,137,3.69,3.18,0.52,75,54.74


In [7]:
output_folder = Path(r"C:\Users\HP\Internship\reports")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "11_Weekday_Category_Analysis.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    weekday_analysis.to_excel(
        writer, sheet_name="Weekday_Analysis", index=False
    )

    category_analysis.to_excel(
        writer, sheet_name="Category_Analysis", index=False
    )

    weekday_category_analysis.to_excel(
        writer, sheet_name="Weekday_Category", index=False
    )

print("Excel file saved successfully!")
print(output_file)

Excel file saved successfully!
C:\Users\HP\Internship\reports\11_Weekday_Category_Analysis.xlsx


In [8]:
print("Weekday rows:", len(weekday_analysis))
print("Category rows:", len(category_analysis))
print("Combined analysis rows:", len(weekday_category_analysis))

print("\nHighest late-delivery weekday:")
print(
    weekday_analysis.sort_values(
        "Late_Delivery_Percentage", ascending=False
    ).head(1)
)

print("\nCategory with highest late-delivery percentage:")
print(category_analysis.head(1))

Weekday rows: 7
Category rows: 50
Combined analysis rows: 337

Highest late-delivery weekday:
  Weekday  Total_Records  Avg_Delivery_Days  Avg_Scheduled_Days  \
0  Monday          25786               3.52                2.94   

   Avg_Delay_Days  Late_Deliveries  On_Time_or_Early  Late_Delivery_Percentage  
0            0.58            14938             10848                     57.93  

Category with highest late-delivery percentage:
        Category Name  Total_Records  Avg_Delivery_Days  Avg_Scheduled_Days  \
23  Golf Bags & Carts             61               3.34                2.57   

    Avg_Delay_Days  Late_Deliveries  On_Time_or_Early  \
23            0.77               42                19   

    Late_Delivery_Percentage  
23                     68.85  
